# PDF Text Extraction

This notebook implements the first stage of the Legal RAG PT pipeline: extracting machine-readable text from the **Porto Municipal Regulatory Code (CRMP)** PDF.

It locates and validates the source document, inspects its metadata and a representative page, extracts text page by page, performs basic quality checks, and saves the result as structured JSON for the preprocessing stage.

## Output

The notebook creates `data/processed/crmp_extracted.json`, containing document metadata and the extracted text for every page.

> Run this notebook from the `notebooks/` directory so that project-relative paths resolve correctly.


## 1. Import the required libraries

Import path utilities, JSON serialization support, and PyMuPDF. These dependencies are used to locate project files, read the PDF, and persist the extracted content in a structured format.


In [2]:
from pathlib import Path
import json
import pymupdf


## 2. Define the input and output paths

Resolve the project root from the current notebook directory, then define the source PDF and destination JSON paths. Printing both paths makes the active configuration explicit before any processing begins.


In [3]:
# The notebook is expected to run from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "crmp_2025-11-27.pdf"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "crmp_extracted.json"

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")


Input:  c:\Users\user\Documents\GitHub\legal-rag-pt\data\raw\crmp_2025-11-27.pdf
Output: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_extracted.json


## 3. Validate the source document

Confirm that the expected PDF exists before attempting extraction. The notebook stops immediately with a clear error if the file is missing, preventing later cells from failing with less informative messages.


In [4]:
# Fail early with a clear message if the source document is unavailable.
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"PDF nÃ£o encontrado: {INPUT_FILE}")

print("PDF encontrado.")


PDF encontrado.


## 4. Inspect the PDF

Open the document with PyMuPDF and display its page count and embedded metadata. This provides an initial integrity check and useful context about the source file.


In [5]:
document = pymupdf.open(INPUT_FILE)

print(f"NÃºmero de pÃ¡ginas: {len(document)}")
print(f"Metadata: {document.metadata}")


Número de páginas: 662
Metadata: {'format': 'PDF 1.4', 'title': 'Código Regulamentar do Município do Porto', 'author': 'susanacunha', 'subject': '', 'keywords': '', 'creator': 'Microsoft® Word 2013', 'producer': 'Microsoft® Word 2013', 'creationDate': "D:20251205175726+00'00'", 'modDate': "D:20251205175726+00'00'", 'trapped': '', 'encryption': None}


## 5. Test text extraction on a sample page

Extract and display the plain text from page 22. Inspecting a representative page helps verify that the PDF contains a usable text layer before processing the full document.


In [6]:
# PyMuPDF uses zero-based indexing, so index 21 corresponds to PDF page 22.
page = document[21]
text = page.get_text("text")

print(text)


 
 
 
Código Regulamentar do Município do Porto | Parte A | A.1. Princípios gerais  
                       22 
 
 
Parte A 
Parte Geral  
Código Regulamentar do Município do Porto 
 
PARTE A 
Parte geral 
 
Artigo A/1.º 
Objeto do código 
1 – O presente código consagra as disposições regulamentares com eficácia externa em 
vigor na área do Município do Porto nos seguintes domínios: 
a) Urbanismo;                                                          
b) Ambiente; 
c) Gestão do espaço público; 
d) Intervenção municipal sobre o exercício de atividades privadas; 
e) Gestão de recursos; 
f) Taxas e outras receitas municipais; 
g) Fiscalização e sancionamento de infrações. 
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições 
regulamentares complementares ao presente código, nele devidamente referenciadas.  
 
Artigo A/2.º 
Objeto da Parte A 
A Parte A consagra:  
a) No Título I, os princípios gerais inspiradores do código, que, para além dos 
princíp

## 6. Define the page-level extraction function

Create a reusable function that opens a PDF safely, iterates over all pages, extracts plain text, trims surrounding whitespace, and stores each page with a human-readable one-based page number.


In [7]:
def extract_pdf(pdf_path):
    pages = []

    # The context manager ensures that the PDF is closed after extraction.
    with pymupdf.open(pdf_path) as document:
        for page_number, page in enumerate(document, start=1):
            text = page.get_text("text")

            # Store one-based page numbers to match the document as read by users.
            pages.append({
                "page": page_number,
                "text": text.strip()
            })

    return pages


## 7. Extract the complete document

Run the extraction function on the CRMP source PDF and report the number of processed pages. The resulting list is the core page-level dataset used throughout the remaining checks.


In [8]:
pages = extract_pdf(INPUT_FILE)

print(f"PÃ¡ginas extraÃ­das: {len(pages)}")


Páginas extraídas: 662


## 8. Preview the extracted pages

Display the beginning of the first three extracted pages. This quick manual review helps detect obvious issues such as missing text, unexpected encoding, or incorrect reading order.


In [9]:
# Limit the preview to keep notebook output concise.
for page in pages[:3]:
    print("=" * 80)
    print(f"PAGE {page['page']}")
    print("=" * 80)
    print(page["text"][:1000])
    print()


PAGE 1
1 
 
 
 
Código Regulamentar do Município do Porto 
 
 
 
Índice 
Preâmbulo……………………………………………………………………………………………….…..03 
Lei Habilitante do CRMP……………………………………………………………………….……….….15 
PARTE A – Parte Geral .................................................................................................................. 22 
A.1. Princípios gerais ....................................................................................................................... 22 
A.2. Disposições comuns ................................................................................................................. 26 
PARTE B - Urbanismo .................................................................................................................... 35 
B.1. Edificação e urbanização .......................................................................................................... 35 
B.2. Toponímia e numeração de edifícios .............................................................

## 9. Identify pages without text

Find pages whose extracted content is empty after whitespace removal. Empty pages may be intentional, image-only, or evidence of extraction problems and should be reviewed before downstream processing.


In [10]:
# Empty pages may indicate image-only content or an extraction issue.
empty_pages = [
    page["page"]
    for page in pages
    if not page["text"].strip()
]

print(f"PÃ¡ginas sem texto: {len(empty_pages)}")
print(empty_pages)


Páginas sem texto: 0
[]


## 10. Calculate corpus-level statistics

Measure the total number of pages, characters, and whitespace-delimited words. These baseline statistics are useful for validating later preprocessing steps and detecting unexpected data loss.


In [11]:
total_characters = sum(len(page["text"]) for page in pages)
total_words = sum(len(page["text"].split()) for page in pages)

print(f"PÃ¡ginas:     {len(pages):,}")
print(f"Caracteres:  {total_characters:,}")
print(f"Palavras:    {total_words:,}")


Páginas:     662
Caracteres:  1,417,550
Palavras:    217,803


## 11. Build the structured output

Assemble document metadata and the extracted page records into a single serializable object. This schema preserves the source filename, page count, and page boundaries needed by later pipeline stages.


In [12]:
# Keep page boundaries so downstream stages can retain source references.
output = {
    "document": "CÃ³digo Regulamentar do MunicÃ­pio do Porto",
    "source_file": INPUT_FILE.name,
    "num_pages": len(pages),
    "pages": pages
}


## 12. Save the extracted dataset

Create the destination directory when necessary and write the structured data as UTF-8 JSON. Non-ASCII characters are preserved so that Portuguese legal text remains readable.


In [13]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        output,
        f,
        ensure_ascii=False,  # Preserve Portuguese characters in readable form.
        indent=2
    )

print(f"Ficheiro criado: {OUTPUT_FILE}")


Ficheiro criado: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_extracted.json


## 13. Verify the saved file

Reload the generated JSON and print its top-level metadata. Reading the file back confirms that serialization succeeded and that the expected fields are available.


In [14]:
# Reload the file to verify that the saved JSON is valid and complete.
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(data["document"])
print(data["source_file"])
print(data["num_pages"])


Código Regulamentar do Município do Porto
crmp_2025-11-27.pdf
662


## 14. Inspect a saved page

Display a larger excerpt from page 22 using the reloaded JSON data. This final check verifies that page text survives the complete extraction and serialization round trip.


In [15]:
# Confirm that the sample page survived the JSON round trip.
print(data["pages"][21]["text"][:2000])


Código Regulamentar do Município do Porto | Parte A | A.1. Princípios gerais  
                       22 
 
 
Parte A 
Parte Geral  
Código Regulamentar do Município do Porto 
 
PARTE A 
Parte geral 
 
Artigo A/1.º 
Objeto do código 
1 – O presente código consagra as disposições regulamentares com eficácia externa em 
vigor na área do Município do Porto nos seguintes domínios: 
a) Urbanismo;                                                          
b) Ambiente; 
c) Gestão do espaço público; 
d) Intervenção municipal sobre o exercício de atividades privadas; 
e) Gestão de recursos; 
f) Taxas e outras receitas municipais; 
g) Fiscalização e sancionamento de infrações. 
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições 
regulamentares complementares ao presente código, nele devidamente referenciadas.  
 
Artigo A/2.º 
Objeto da Parte A 
A Parte A consagra:  
a) No Título I, os princípios gerais inspiradores do código, que, para além dos 
princípios ge